Convert nielson_conversion.json to nielson_data.csv

In [15]:
# Make the Nielson JSON easy to read as a spreadsheet
from resources.nielson_data import NielsonDataCSVConverter

# Define file paths
json_file = "resources/nielson_ome.json"
csv_file = "resources/nielson_data.csv"

# Create an instance of the converter
converter = NielsonDataCSVConverter(json_file, csv_file)

# Convert the JSON to CSV
converter.convert_to_csv()

JSON data has been successfully converted to resources/nielson_data.csv


Calculate the equivalent opioids from nielson_ome.json

In [16]:
# Conversion Service
from common.opioid import Opioid
from common.conversion_service import ConversionService
from resources.nielson_data import NIELSON_CONVERSIONS

conversion_service = ConversionService(NIELSON_CONVERSIONS)

iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl")
oral_hydromorphone = Opioid(administration_method="oral", substance="hydromorphone")
oral_morphine = Opioid(administration_method="oral", substance="morphine")

random_dose = conversion_service.generate_random_dose(iv_fentanyl)
print(f"Random dose for {iv_fentanyl.administration_method}_{iv_fentanyl.substance}: {random_dose:.2f} {iv_fentanyl.unit}")

equivalent_dose_oral_morphine = conversion_service.calculate_expected_dose(random_dose, iv_fentanyl, oral_morphine)
print(f"Equivalent dose for {oral_morphine.administration_method}_{oral_morphine.substance}: {equivalent_dose_oral_morphine:.2f} {oral_morphine.unit}")

Random dose for iv_fentanyl: 162.35 mcg/day
Equivalent dose for oral_morphine: 32.47 mg/day


## Opioid Pair Experiment - Effect of Including Nielson

Ask the model for the equivalent opioid dose

In [5]:
# Run a single conversion with an instance of PromptedOpioidConversion
from common.conversion_service import ConversionService
from experiment.prompt_service import PromptService, PromptedOpioidConversion
from resources.nielson_data import NIELSON_CONVERSIONS
from common.opioid import Opioid

# Initialize the OpenAI client once
PromptedOpioidConversion.initialize_client()

# Define opioids
iv_fentanyl = Opioid(administration_method="iv", substance="fentanyl", unit="mcg/day")
oral_morphine = Opioid(administration_method="oral", substance="morphine", unit="mg/4 hours")
to_opioid = oral_morphine

# Create a PromptedOpioidConversion object
conversion = PromptedOpioidConversion(
    from_opioid=iv_fentanyl,
    to_opioid=to_opioid,
    model="deepseek/deepseek-chat",
    include_nielson=True,
    include_examples=True,
    ask_for_units=True,
    explain_units=True
)

# Generate the prompt
conversion.generate_prompt(prompt_service=PromptService(ConversionService(NIELSON_CONVERSIONS)))

# Query the model
observed_dose = conversion.ask_model()

# Print the results
print("Generated Prompt:")
print(conversion.prompt)
print(f"Observed Dose: {observed_dose:.2f} {to_opioid.unit}")
print(f"Expected Dose: {conversion.expected_to_dose:.2f} {to_opioid.unit}")
print(f"Ratio Observed/Expected: {conversion.ratio_observed_expected:.2f}")
print("LLM Response Text:")
print(conversion.response) 

Generated Prompt:
Convert 466.25 mcg/day of fentanyl (iv) to mg/4 hours of morphine (oral).
How many mg/4 hours of morphine (oral) is equivalent to 466.25 mcg/day of fentanyl (iv)?

0.2 OME: 1 mcg/day of fentanyl (iv) is equivalent to 0.2 mg/day morphine (oral).
0.2 OME for fentanyl (iv) means 10 mg/day of morphine (oral) is equivalent to 50.00 mcg/day of fentanyl (iv).
And 120 mg/day of morphine (oral) is equivalent to 600.00 mcg/day of fentanyl (iv).

Notice that this is a request for the equivalent dose administered over 4 hours, not over 1 day. 
The equivalent dose over 4 hours should be 1/6 the total dose for 1 day.
Express the response in the format {'calculated dose': value, 'units': 'mg/4 hours'}. Do not include any additional text in your response, only the dictionary.
Observed Dose: 15.54 mg/4 hours
Expected Dose: 15.54 mg/4 hours
Ratio Observed/Expected: 1.00
LLM Response Text:
{'calculated dose': 15.54, 'units': 'mg/4 hours'}


Run an experiment converting opioids across prompting conditions and models

In [ ]:
# Run an experiment converting opioids across prompting conditions and models
from experiment.prompt_service import PromptedOpioidConversion
from common.opioid import OpioidPairGenerator
from experiment.experiment import Experiment
from common.models import CONSISTENT_MODELS

# Initialize all clients
PromptedOpioidConversion.initialize_client()

# Generate opioid pairs
opioid_list_ome = OpioidPairGenerator(lambda priority: priority <= 4 if priority else False)\
    .generate_opioid_list_or_pairs(pairs=False, use_permutations=False)

opioid_pairs = OpioidPairGenerator(lambda priority: priority == 1 if priority else False)\
    .generate_opioid_list_or_pairs(pairs=True, use_permutations=True)

# Define prompt conditions
prompt_conditions = [
    {"include_nielson": False, "include_examples": False},
    {"include_nielson": True, "include_examples": False},
    {"include_nielson": True, "include_examples": True}
]

# OpenRouter models
models = CONSISTENT_MODELS

output_csv = "results/nielson_experiment_results.csv"

# Run OME experiment
experiment_ome = Experiment(opioid_list_ome, prompt_conditions, models, repetitions=25, filename=output_csv)
experiment_ome.run()

# Print the OME experiment ID
print(f"Experiment ID: {experiment_ome.experiment_id}")

# Run pairs experiment
experiment_pairs = Experiment(opioid_pairs, prompt_conditions, models, repetitions=25, filename=output_csv)
experiment_pairs.run()

# Print the pairs experiment ID
print(f"Experiment ID: {experiment_pairs.experiment_id}")

Process results into summaries

In [ ]:
from analysis.analysis_utils import processing_pipeline

# For opioid pair experiments:
processing_pipeline(
    input_csv="nielson_experiment_results.csv",
    output_csv="nielson_experiment_results_by_ome.csv",
    variable_model_csv="nielson_pair_model_analysis.csv",
    consensus_csv="nielson_pair_consensus.csv",
    mode="pair"
)

Summarize and name batch means

In [1]:
# Make a CSV of model batch means for weight, tight, and broad ranges

from analysis.model_means import ModelBatchMeansBuilder

combinations = [
    {"ome": None, "include_nielson": False, "include_examples": False, "batch_name": "all_n0"},
    {"ome": None, "include_nielson": True, "include_examples": False, "batch_name": "all_n1"},
    {"ome": True, "include_nielson": False, "include_examples": False, "batch_name": "ome_n0"},
    {"ome": True, "include_nielson": True, "include_examples": False, "batch_name": "ome_n1"},
    {"ome": False, "include_nielson": False, "include_examples": False, "batch_name": "pairs_n0"},
    {"ome": False, "include_nielson": True, "include_examples": False, "batch_name": "pairs_n1"},
]

builder = ModelBatchMeansBuilder("results/pair_model_analysis.csv")
builder.save_means_df(combinations, "results/model_batch_means.csv")

✓ Saved batch means to results/model_batch_means.csv


# Visualize Naive Results and Effect of Including Nielson

Visualize a single opioid pair

In [11]:
from visualize.opioid_pair_visualizer import OpioidPairVisualizer
from bokeh.io import show
from common.opioid_unit_finder import OpioidUnitFinder

# Change to a different opioid pair
from_opioid = "oral_oxycodone"
to_opioid = "oral_morphine"
prompt_conditions = {"include_nielson": False, "include_examples": False}

# Create the visualizer instance
unit_finder = OpioidUnitFinder("results/experiment_results_by_ome.csv")
from_opioid_unit = unit_finder.get_unit("from_opioid", from_opioid, "from_opioid_unit")
to_opioid_unit = unit_finder.get_unit("to_opioid", to_opioid, "to_opioid_unit")

visualizer = OpioidPairVisualizer(
    experiment_results_csv="results/experiment_results_by_ome.csv",
    pair_model_csv="results/pair_model_analysis.csv",
    consensus_csv="results/opioid_pair_consensus.csv",
    from_opioid=from_opioid,
    from_opioid_unit=from_opioid_unit,
    to_opioid=to_opioid,
    to_opioid_unit=to_opioid_unit,
    prompt_conditions=prompt_conditions
)

# Show the visualization
layout = visualizer.create_plot(width=600, height=350)
show(layout)

In [3]:
# Use the current version of the heatmap visualizer to make a bar chart (no updates)

from analysis.batch import OpioidPairBatch
from visualize.opioid_pair_batch_heatmap_visualizer import OpioidPairBatchHeatmapVisualizer
from bokeh.plotting import show

# If you want to use the current method, you need to provide a batch object
# Example: create a batch for the desired batch_name
batch_name = "ome_n1"  # Change to your desired batch

# You may need to construct filter_specs to match your batch_name logic
filter_spec = {
    "include_nielson": True,  # Set according to batch_name
    "include_examples": False,
    "exclude_self_pairs": True,
    "ome": True  # Set according to batch_name
}
filter_specs = [filter_spec]

batch = OpioidPairBatch.from_filter_specs(
    experiment_results_csv="results/experiment_results_by_ome.csv",
    variable_model_csv="results/pair_model_analysis.csv",
    consensus_csv="results/opioid_pair_consensus.csv",
    filter_specs=filter_specs,
)

heatmap_vis = OpioidPairBatchHeatmapVisualizer(batch)
bar_chart = heatmap_vis._create_mean_accuracy_bar_chart(width=900, height=400)
show(bar_chart)

Found 16 models and 19 pairs
Weight matrix shape: (17, 20)
Non-NaN values in weight matrix: 0


/Users/tanyamarton/opioid-conversion-experiment/analysis/batch.py:101: RuntimeWarning: Mean of empty slice
  col_means = np.nanmean(matrix, axis=1, keepdims=True)
/Users/tanyamarton/opioid-conversion-experiment/analysis/batch.py:103: RuntimeWarning: Mean of empty slice
  row_means = np.nanmean(matrix, axis=0, keepdims=True)
/Users/tanyamarton/opioid-conversion-experiment/analysis/batch.py:175: RuntimeWarning: Mean of empty slice
  tight = np.nanmean(tight_matrix[i, :-1])
/Users/tanyamarton/opioid-conversion-experiment/analysis/batch.py:176: RuntimeWarning: Mean of empty slice
  broad = np.nanmean(broad_matrix[i, :-1])


Visualize and save pdfs of all batches and heatmaps

In [ ]:
from analysis.batch import OpioidPairBatch
from visualize.batch_visualizer import OpioidPairBatchVisualizer
from visualize.opioid_pair_batch_heatmap_visualizer import OpioidPairBatchHeatmapVisualizer
from visualize.visualization_config import convert_to_pdf_auto_size

combinations = [
   {"ome": None, "include_nielson": False, "include_examples": False, "name": "all_n0"},
   {"ome": None, "include_nielson": True, "include_examples": False, "name": "all_n1"},    
   {"ome": True, "include_nielson": False, "include_examples": False, "name": "ome_n0"},
   {"ome": True, "include_nielson": True, "include_examples": False, "name": "ome_n1"},  
   {"ome": False, "include_nielson": False, "include_examples": False, "name": "pairs_n0"},
   {"ome": False, "include_nielson": True, "include_examples": False, "name": "pairs_n1"},  
]

for combo in combinations:
    filter_spec = {
        "include_nielson": combo["include_nielson"],
        "include_examples": False,
        "exclude_self_pairs": True,
    }
    if combo["ome"] is not None:
        filter_spec["ome"] = combo["ome"]
    filter_specs = [filter_spec]
    
    try:
        batch = OpioidPairBatch.from_filter_specs(
            experiment_results_csv="results/experiment_results_by_ome.csv",
            variable_model_csv="results/pair_model_analysis.csv",
            consensus_csv="results/opioid_pair_consensus.csv",
            filter_specs=filter_specs,
        )

        vis = OpioidPairBatchVisualizer(
            batch,
            row_width=600,
            row_height=300,
        )

        batch_html = f"figures/{combo['name']}_batch.html"

        vis.create_grid(output_html=batch_html)
        print(f"✓ Saved Bokeh HTML to {batch_html}")
        
        # Convert Bokeh to PDF
        batch_pdf = f"figures/{combo['name']}_batch.pdf"
        await convert_to_pdf_auto_size(batch_html, batch_pdf, scale=2.0)
        print(f"✓ Saved Bokeh PDF to {batch_pdf}")

        heatmap_vis = OpioidPairBatchHeatmapVisualizer(
            batch,
            cell_size=35,
            width_padding=0,
            height_padding=0,
            cell_text_size="6pt",
            label_text_size="7pt"
        )

        # Save HTML
        output_html = f"figures/{combo['name']}_heatmaps.html"
        heatmap_vis.create_heatmaps(output_html=output_html)
        print(f"✓ Saved HTML to {output_html}")
        
        # Convert to PDF
        output_pdf = f"figures/{combo['name']}_heatmaps.pdf"
        await convert_to_pdf_auto_size(output_html, output_pdf, scale=3.0)
        print(f"✓ Saved PDF to {output_pdf}")

    except Exception as e:
        print(f"✗ Error creating {combo['name']}: {e}")

Visualize summary plots

In [ ]:
from visualize.opioid_model_comparison_bar_visualizer import ModelComparisonFigureBuilder

builder = ModelComparisonFigureBuilder(batch_means_csv="results/model_batch_means.csv")
await builder.save("figures/model_comparison_bar.html", "figures/model_comparison_bar.pdf")
builder.save_scatter_figures_pdf("figures/model_comparison_scatter.pdf") 

✓ Saved HTML to figures/explain_units_bar.html
✓ Generated PDF: 1300px x 740px
✓ Saved PDF to figures/explain_units_bar.pdf


# Opioid Pair (Non-Nielson Units) Experiment

In [ ]:
from common.opioid import Opioid
from common.opioid import OpioidPairGenerator
from experiment.prompt_service import PromptedOpioidConversion
from experiment.experiment import Experiment

# Generate OME opioids (priority <= 2)
from_opioids = OpioidPairGenerator(lambda priority: priority <= 4 if priority else False)\
    .generate_opioid_list_or_pairs(pairs=False, use_permutations=False)

# Create oral morphine with custom unit
to_opioid = Opioid(administration_method="oral", substance="morphine", unit="mg/4 hours")

# Build pairs: each from_opioid paired with custom-unit oral morphine
opioid_pairs = [(from_opioid, to_opioid) for from_opioid in from_opioids]

# Models to use
models = CONSISTENT_MODELS

# Prompt conditions: always ask for units, always include Nielson
prompt_conditions = [
    {"include_nielson": True, "include_examples": False, "ask_for_units": True, "explain_units": True},
    {"include_nielson": True, "include_examples": False, "ask_for_units": True, "explain_units": False},
]

# Initialize client
PromptedOpioidConversion.initialize_client()

# Run the experiment and append results
experiment = Experiment(
    opioid_pairs,
    prompt_conditions,
    models,
    repetitions=25,
    filename="results/explain_units_experiment_results.csv"
)
experiment.run()

print(f"Experiment ID: {experiment.experiment_id}")

Process data for units experiment

In [ ]:
from analysis.analysis_utils import processing_pipeline

# For opioid pair experiments:
processing_pipeline(
    input_csv="results/explain_units_experiment_results.csv",
    output_csv="results/explain_units_experiment_results_by_ome.csv",
    variable_model_csv="results/explain_units_pair_model_analysis.csv",
    consensus_csv="results/explain_units_opioid_pair_consensus.csv",
    mode="pair"
)

In [8]:
from analysis.model_means import ModelBatchMeansBuilder
combinations = [
    {"explain_units": False, "batch_name": "4hours_units"},
    {"explain_units": True, "batch_name": "4hours_units_explained"}
]
builder = ModelBatchMeansBuilder("results/explain_units_pair_model_analysis.csv")
builder.save_means_df(combinations, "results/model_batch_means_explain_units.csv")

✓ Saved batch means to results/model_batch_means_explain_units.csv


Visualize results from units experiment

In [1]:
from visualize.explain_units_visualizer import ExplainUnitsVisualizer
from common.models import CONSISTENT_MODELS

vis = ExplainUnitsVisualizer(
    batch_means_csv="results/model_batch_means.csv",
    batch_means_explain_units_csv="results/model_batch_means_explain_units.csv",
    models = CONSISTENT_MODELS
)
await vis.save("figures/explain_units_bar.html", "figures/explain_units_bar.pdf")


✓ Saved HTML to figures/explain_units_bar.html
✓ Generated PDF: 1300px x 740px
✓ Saved PDF to figures/explain_units_bar.pdf


## Scenario Experiment

Construct a scenario prompt

In [3]:
from experiment.prompt_service import PromptService
from common.conversion_service import ConversionService
from resources.nielson_data import NIELSON_CONVERSIONS
from resources.scenario_vignettes import SCENARIO_VIGNETTES
from experiment.scenario import Scenario, ScenarioPromptedConversion

# Initialize the PromptService
prompt_service = PromptService(ConversionService(NIELSON_CONVERSIONS))

# Select a scenario by ID (e.g., "v2")
scenario_dict = SCENARIO_VIGNETTES["v81"]
scenario = Scenario(scenario_dict)

# Create a ScenarioPromptedConversion object
scenario_conversion = ScenarioPromptedConversion(
    scenario,
    model="google/gemini-2.5-pro",
    include_nielson=True,
    include_examples=False,
    ask_for_units=True,
    explain_units=True
)

# Generate the prompt using the PromptService instance
generated_prompt = scenario_conversion.generate_prompt(prompt_service)

# Print the generated prompt
print("Generated Prompt:")
print(generated_prompt)

# Print the sampled dose and calculated expected_to_dose
print(f"Expected Dose: {scenario_conversion.expected_to_dose:.2f} {scenario_conversion.to_opioid.unit}")

Generated Prompt:
A woman with metastatic breast cancer and ongoing pain management needs receives oral hydromorphone in divided doses totaling 24 mg daily. As part of long-term analgesic planning, calculate the equivalent transdermal fentanyl patch dose.

Convert 24.00 mg/day of hydromorphone (oral) to mcg/hr of fentanyl (transdermal).
How many mcg/hr of fentanyl (transdermal) is equivalent to 24.00 mg/day of hydromorphone (oral)?

5.0 OME: 1 mg/day of hydromorphone (oral) is equivalent to 5.0 mg/day morphine (oral).
2.7 OME: 1 mcg/hr of fentanyl (transdermal) is equivalent to 2.7 mg/day morphine (oral).


Express the response in the format {'calculated dose': value, 'units': 'mcg/hr'}. Do not include any additional text in your response, only the dictionary.
Expected Dose: 44.44 mcg/hr


Run an experiment using scenarios

In [ ]:
from resources.scenario_vignettes import SCENARIO_VIGNETTES
from experiment.scenario import Scenario, ScenarioPromptedConversion
from experiment.scenario_experiment import ScenarioExperiment
from experiment.prompt_service import PromptedOpioidConversion
from common.models import CONSISTENT_MODELS

PromptedOpioidConversion.initialize_client()

# Vignette IDs to test
vignette_ids = ["v"+str(i) for i in range(2, 82)]

# Create ScenarioPromptedConversion objects for each vignette
scenarios = [
    ScenarioPromptedConversion(
        Scenario(SCENARIO_VIGNETTES[v_id]),
        model=None,  # Will be set by experiment
        include_nielson=False
    )
    for v_id in vignette_ids
]

# Prompt conditions: no Nielson, no examples
prompt_conditions = [
    #{"include_nielson": False, "include_examples": False},
    #{"include_nielson": True, "include_examples": False},
    #{"include_nielson": True, "include_examples": True},
    {"include_nielson": True, "include_examples": False, "ask_for_units": True, "explain_units": True},
    #{"include_nielson": True, "include_examples": True, "ask_for_units": True, "explain_units": True}
]

# All models to test
models = CONSISTENT_MODELS

# Specify output CSV filename
output_csv = "results/reworded_vignettes_results.csv"

# Run the experiment (one repetition per scenario/model)
experiment = ScenarioExperiment(
    scenarios=scenarios,
    prompt_conditions=prompt_conditions,
    models=models,
    repetitions=7,
    filename=output_csv
)
experiment.run()

Error during scenario conversion: Could not extract numeric value from response: 
Error during scenario conversion: Could not extract numeric value from response: 
Error during scenario conversion: Could not extract numeric value from response: 
Error during scenario conversion: Could not extract numeric value from response: 
Error during scenario conversion: Could not extract numeric value from response: 
Error during scenario conversion: Could not extract numeric value from response: 
Error during scenario conversion: Could not extract numeric value from response: 
Error during scenario conversion: OpenRouter API error for model cohere/command-r-plus-08-2024: 'NoneType' object is not subscriptable
Error during scenario conversion: OpenRouter API error for model cohere/command-r-plus-08-2024: 'NoneType' object is not subscriptable
Error during scenario conversion: Could not extract numeric value from response: 
Error during scenario conversion: Could not extract numeric value from res

Process Scenario Data

In [4]:
from analysis.analysis_utils import processing_pipeline
from analysis.scenario_model_result import ScenarioModelResult

# For scenario experiments:
processing_pipeline(
    input_csv="results/scenario_experiment_results.csv",
    output_csv="results/scenario_experiment_results_by_ome.csv",
    variable_model_csv="results/vignette_model_analysis.csv",
    consensus_csv="results/vignette_model_consensus.csv",
    mode="vignette"
)

ScenarioModelResult.process_all_combinations(
    vignette_results_csv="results/vignette_model_analysis.csv",
    output_file="results/scenario_model_analysis.csv",
    verbose=True
)

Added 'ome' column to results/scenario_experiment_results_by_ome.csv
Harmonized model names written to results/scenario_experiment_results_by_ome.csv
Processed 50/560 combinations...
Processed 100/560 combinations...
Processed 150/560 combinations...
Processed 200/560 combinations...
Processed 250/560 combinations...
Processed 300/560 combinations...
Processed 350/560 combinations...
Processed 400/560 combinations...
Processed 450/560 combinations...
Processed 500/560 combinations...
Processed 550/560 combinations...
Completed! Processed 560/560 combinations.
Results saved to: results/vignette_model_analysis.csv
Completed! Processed 40/40 consensus groups.
Consensus results saved to: results/vignette_model_consensus.csv
Processed 50/140 scenario groups...
Processed 100/140 scenario groups...
Completed! Processed 140/140 scenario groups.
Scenario results saved to: results/scenario_model_analysis.csv


[{'scenario_key': 'middle-aged-woman-cancer-surgery',
  'from_opioid': 'oral_hydromorphone,iv_hydromorphone',
  'from_opioid_unit': 'mg/day,mg/day',
  'to_opioid': 'oral_hydromorphone',
  'to_opioid_unit': 'mg/4hr',
  'multiple_from_opioids': np.True_,
  'ome': np.False_,
  'model': 'ai21/jamba-large-1.7',
  'include_nielson': np.False_,
  'include_examples': np.False_,
  'ask_for_units': np.False_,
  'explain_units': np.False_,
  'observed_log_dose_mean': 1.5628845452147808,
  'observed_dose_mean': 4.772568097365081,
  'observed_log_dose_within_std': 0.06190859095471684,
  'observed_log_dose_between_std': 0.7671934793884092,
  'observed_log_dose_std': 0.7696872796468003,
  'observed_log_dose_between_cv': 0.49088301611106977,
  'effective_n_vignettes': 3.9815807755057615,
  'n_vignettes': 4,
  'total_observations': 28.0,
  'tight(0.85,1.15)': 0.0,
  'broad(0.6,1.7)': 0.0357142857142857},
 {'scenario_key': 'middle-aged-woman-cancer-surgery',
  'from_opioid': 'oral_hydromorphone,iv_hydro

In [ ]:
from analysis.model_means import ModelBatchMeansBuilder
combinations = [
    {"include_nielson": False, "include_examples": False, 'ask_for_units': False, 'explain_units': False, 'batch_name': 'naive'},
    {"include_nielson": True, "include_examples": False, 'ask_for_units': False, 'explain_units': False, 'batch_name': 'nielson_only'},
    {"include_nielson": True, "include_examples": True, 'ask_for_units': False, 'explain_units': False, 'batch_name': 'nielson_examples'},
    {"include_nielson": True, "include_examples": False, "ask_for_units": True, "explain_units": True, 'batch_name': 'nielson_units_explained'},
    {"include_nielson": True, "include_examples": True, "ask_for_units": True, "explain_units": True, 'batch_name': 'nielson_examples_units_explained'}
]
builder = ModelBatchMeansBuilder("results/scenarior_model_analysis.csv")
builder.save_means_df(combinations, "results/model_batch_means_explain_units.csv")

# Visualize Vignette Results

Visualize a single vignette result

In [ ]:
# Example usage for vignette visualization

from visualize.vignette_visualizer import VignetteVisualizer
from bokeh.io import show

# Set up your file paths and parameters
experiment_results_csv = "scenario_experiment_results_by_ome.csv"
vignette_model_csv = "scenario_model_analysis.csv"
consensus_csv = "vignette_model_consensus.csv"
vignette_id = "v2"  # Change to your vignette
prompt_conditions = {"include_nielson": False, "include_examples": False}

# Create the visualizer instance
visualizer = VignetteVisualizer(
    experiment_results_csv=experiment_results_csv,
    vignette_model_csv=vignette_model_csv,
    consensus_csv=consensus_csv,
    vignette_id=vignette_id,
    prompt_conditions=prompt_conditions,
)

# Show the visualization
layout = visualizer.create_plot(width=650, height=300)
show(layout)

Visualize all vignette results as a batch

In [ ]:
from analysis.batch import VignetteBatch
from visualize.batch_visualizer import VignetteBatchVisualizer
from visualize.visualization_config import convert_to_pdf_auto_size

# Define filter specs for all vignettes and prompt conditions
filter_specs = [
    {"include_nielson": True, "include_examples": False},
    {"include_nielson": False, "include_examples": False},
]

# Create the batch (adjust file paths as needed)
batch = VignetteBatch.from_filter_specs(
    experiment_results_csv="scenario_experiment_results_by_ome.csv",
    variable_model_csv="scenario_model_analysis.csv",
    consensus_csv="vignette_model_consensus.csv",
    filter_specs=filter_specs,
)

# Visualize all combinations in one HTML
vis = VignetteBatchVisualizer(
    batch,
    row_width=900,
    row_height=300,
)
output_html = "all_vignette_batch.html"
vis.create_grid(output_html=output_html)

# Convert to PDF
output_pdf = "all_vignette_batch.pdf"
await convert_to_pdf_auto_size(output_html, output_pdf, scale=2.0)
print(f"✓ Saved batch PDF to {output_pdf}")

In [2]:
from visualize.vignette_scatter_visualizer import VignetteScatterVisualizer
from visualize.visualization_config import convert_to_pdf_auto_size
from bokeh.io import show, output_file, save

# Create and show the scatter plot
vis = VignetteScatterVisualizer("results/vignette_model_analysis.csv")
layout = vis.create_layout()
show(layout)

# Save as HTML and convert to PDF
output_html = "figures/vignette_scatter.html"
output_pdf = "figures/vignette_scatter.pdf"
output_file(output_html)
save(layout)
await convert_to_pdf_auto_size(output_html, output_pdf, scale=2.0)

/Users/tanyamarton/opioid-conversion-experiment/visualize/vignette_scatter_visualizer.py:29: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  base_df = sub_df[self.cond_base][['vignette_id', 'model', self.tight_col]]
/Users/tanyamarton/opioid-conversion-experiment/visualize/vignette_scatter_visualizer.py:30: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  units_df = sub_df[self.cond_units][['vignette_id', 'model', self.tight_col]]
/Users/tanyamarton/opioid-conversion-experiment/visualize/vignette_scatter_visualizer.py:29: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  base_df = sub_df[self.cond_base][['vignette_id', 'model', self.tight_col]]
/Users/tanyamarton/opioid-conversion-experiment/visualize/vignette_scatter_visualizer.py:30: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  units_df = sub_df[self.cond_units][['vignette_id', 'model', self.tight_col]]


✓ Generated PDF: 1300px x 998px


In [ ]:
merged

12.444444444444445